# 03 Baselines And Split
Этап 8: подготовка df_model, честный split по CSBID, baseline-эвристики и сохранение split-артефактов.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from _shared_notebook_utils import RESEARCH_CHECKPOINT_DIR, DATA_DIR, ensure_dirs, save_pickle, save_json, assert_no_group_leakage

ensure_dirs()

## 1) Загрузка window dataset

In [2]:
window_path = DATA_DIR / '02_window_df_filtered.pkl'
if not window_path.exists():
    fallback_path = RESEARCH_CHECKPOINT_DIR / 'baseline' / 'window_df_filtered.pkl'
    if not fallback_path.exists():
        raise FileNotFoundError('No window dataset found. Run notebook 02 or ensure checkpoint fallback exists.')
    df_model = pd.read_pickle(fallback_path)
    source = 'fallback_checkpoint_window_df_filtered'
else:
    df_model = pd.read_pickle(window_path)
    source = '02_window_df_filtered.pkl'

required_cols = ['CSBID', 'history_1', 'history_2', 'history_3', 'target']
missing = [c for c in required_cols if c not in df_model.columns]
if missing:
    raise ValueError(f'Missing required columns in df_model: {missing}')

print('source =', source)
print('df_model shape =', df_model.shape)

source = 02_window_df_filtered.pkl
df_model shape = (38511210, 11)


## 2) Кодирование target

In [3]:
classes_in_order = sorted(df_model['target'].astype('string').dropna().unique().tolist())
target_to_id = {name: idx for idx, name in enumerate(classes_in_order)}
id_to_target = {idx: name for name, idx in target_to_id.items()}

df_model['target_encoded'] = df_model['target'].astype('string').map(target_to_id).astype(int)
print('n_classes =', len(classes_in_order))
print('classes =', classes_in_order)

n_classes = 9
classes = ['corn', 'cotton', 'fallow', 'forage_hay', 'legumes', 'other_cereals', 'sorghum', 'soybeans', 'wheat']


## 3) Honest split по CSBID

In [4]:
RANDOM_STATE = 42
TRAIN_SHARE = 0.7
VAL_SHARE = 0.15
TEST_SHARE = 0.15

if abs((TRAIN_SHARE + VAL_SHARE + TEST_SHARE) - 1.0) > 1e-9:
    raise ValueError('Split shares must sum to 1.0')

field_ids = pd.Series(df_model['CSBID'].dropna().unique()).astype('string')
train_ids, temp_ids = train_test_split(field_ids, test_size=(1 - TRAIN_SHARE), random_state=RANDOM_STATE)
rel_test = TEST_SHARE / (VAL_SHARE + TEST_SHARE)
val_ids, test_ids = train_test_split(temp_ids, test_size=rel_test, random_state=RANDOM_STATE)

train_id_set = set(train_ids.astype('string'))
val_id_set = set(val_ids.astype('string'))
test_id_set = set(test_ids.astype('string'))

key = df_model['CSBID'].astype('string')
train_df = df_model.loc[key.isin(train_id_set)].copy()
val_df = df_model.loc[key.isin(val_id_set)].copy()
test_df = df_model.loc[key.isin(test_id_set)].copy()

assert_no_group_leakage(train_ids, val_ids, test_ids)

print('train_df:', train_df.shape)
print('val_df:', val_df.shape)
print('test_df:', test_df.shape)

train_df: (26957630, 12)
val_df: (5776566, 12)
test_df: (5777014, 12)


## 4) Быстрые baseline-метрики

In [5]:
most_freq_target = train_df['target'].astype('string').value_counts().index[0]

def evaluate_simple(split_df, split_name):
    y_true = split_df['target'].astype('string')
    pred_mf = pd.Series(most_freq_target, index=split_df.index)
    pred_last = split_df['history_3'].astype('string')
    return [
        {
            'model': 'most_frequent',
            'split': split_name,
            'accuracy': float(accuracy_score(y_true, pred_mf)),
            'macro_f1': float(f1_score(y_true, pred_mf, average='macro', zero_division=0))
        },
        {
            'model': 'last_crop',
            'split': split_name,
            'accuracy': float(accuracy_score(y_true, pred_last)),
            'macro_f1': float(f1_score(y_true, pred_last, average='macro', zero_division=0))
        }
    ]

baseline_results = pd.DataFrame(evaluate_simple(val_df, 'validation') + evaluate_simple(test_df, 'test'))
display(baseline_results.sort_values(['split', 'macro_f1'], ascending=[True, False]))

,model,split,accuracy,macro_f1
3,last_crop,test,0.354321,0.172575
2,most_frequent,test,0.305068,0.051946
1,last_crop,validation,0.354267,0.172492
0,most_frequent,validation,0.305306,0.051977


## 5) Сохранение split-артефактов

In [ ]:
baseline_dir = RESEARCH_CHECKPOINT_DIR / 'baseline'
baseline_dir.mkdir(parents=True, exist_ok=True)

save_pickle(df_model, baseline_dir / 'df_model.pkl')
save_pickle(train_df, baseline_dir / 'train_df.pkl')
save_pickle(val_df, baseline_dir / 'val_df.pkl')
save_pickle(test_df, baseline_dir / 'test_df.pkl')
save_pickle(train_ids.reset_index(drop=True), baseline_dir / 'train_ids.pkl')
save_pickle(val_ids.reset_index(drop=True), baseline_dir / 'val_ids.pkl')
save_pickle(test_ids.reset_index(drop=True), baseline_dir / 'test_ids.pkl')

meta = {
    'split_source': source,
    'random_state': RANDOM_STATE,
    'train_share': TRAIN_SHARE,
    'val_share': VAL_SHARE,
    'test_share': TEST_SHARE,
    'target_to_id': target_to_id,
    'id_to_target': {str(k): v for k, v in id_to_target.items()},
    'n_rows': int(df_model.shape[0]),
    'n_classes': int(len(target_to_id))
}

save_json(meta, RESEARCH_CHECKPOINT_DIR / 'baseline_meta.json')
save_pickle(baseline_results, DATA_DIR / '03_baseline_results.pkl')

print('Saved baseline checkpoint to:', baseline_dir)
print('Saved meta:', RESEARCH_CHECKPOINT_DIR / 'baseline_meta.json')

Saved baseline checkpoint to: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\research_checkpoint\baseline
Saved meta: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\research_checkpoint\baseline_meta.json


: 